# Phase 2: Axial Coding - Failure Taxonomy Generation

**Goal:** Organize open codes from Phase 1 into coherent failure categories based on workflow stages.

**Methodology:** Workflow-stage taxonomy with merged related failures

**Exclusions:** Technical errors (token_limit) - treated separately

---

## Step 1: Load and Explore Phase 1 Data

In [ ]:
import json
import pandas as pd
from collections import Counter, defaultdict
import matplotlib.pyplot as plt
import seaborn as sns

# Load annotations
annotations = []
with open('traces/traces_annotations.jsonl', 'r') as f:
    for line in f:
        if line.strip():
            annotations.append(json.loads(line))

print(f"Total annotations: {len(annotations)}")

# Display first annotation
if annotations:
    print("\nSample annotation:")
    print(json.dumps(annotations[0], indent=2))

In [ ]:
# Filter out token_limit errors (technical/environmental, solved)
filtered_annotations = []
for ann in annotations:
    modes = ann.get('open_coded_failure_modes', [])
    # Keep only if not purely token_limit
    if modes and 'token_limit' not in modes or not modes:
        # But remove token_limit from the list if it's mixed
        cleaned_modes = [m for m in modes if 'token_limit' not in m and m != 'toke_limit']
        ann['open_coded_failure_modes'] = cleaned_modes
        # Keep annotation if it has other modes or is PASS
        if cleaned_modes or ann.get('verdict') == 'PASS':
            filtered_annotations.append(ann)
    elif not modes:
        # Keep PASS traces even with no modes
        filtered_annotations.append(ann)

print(f"Annotations after filtering token_limit: {len(filtered_annotations)}")

# Verdict distribution
pass_count = sum(1 for a in filtered_annotations if a.get('verdict') == 'PASS')
fail_count = sum(1 for a in filtered_annotations if a.get('verdict') == 'FAIL')

print(f"\nVerdict Distribution:")
print(f"  PASS: {pass_count}")
print(f"  FAIL: {fail_count}")

## Step 2: Extract and Normalize Open Codes

In [ ]:
# Collect all failure modes
all_modes = []
mode_comments = defaultdict(list)

for ann in filtered_annotations:
    modes = ann.get('open_coded_failure_modes', [])
    comment = ann.get('comment', '')
    
    for mode in modes:
        # Normalize mode names (trim, lowercase)
        normalized = mode.strip().lower()
        all_modes.append(normalized)
        mode_comments[normalized].append({
            'run_id': ann.get('run_id', '?')[:8],
            'comment': comment[:100] + '...' if len(comment) > 100 else comment,
            'verdict': ann.get('verdict')
        })

# Count frequencies
mode_counts = Counter(all_modes)

print("Open Codes (from Phase 1):")
print("="*60)
for mode, count in mode_counts.most_common():
    print(f"{mode:40} : {count:2d} occurrence(s)")

print(f"\nTotal unique modes: {len(mode_counts)}")
print(f"Total mode occurrences: {sum(mode_counts.values())}")

## Step 3: Define Workflow-Based Taxonomy

Organizing failures by PatentDiff workflow stage with consolidated categories.

In [ ]:
# Define taxonomy: Map open codes to consolidated categories by workflow stage
failure_taxonomy = {
    "STAGE_1_INPUT_UNDERSTANDING": {
        "name": "Input Understanding & Decomposition",
        "description": "Failures in understanding input patents, decomposing claims, and identifying preprocessing steps",
        "categories": {
            "Claim Decomposition & Structure": {
                "consolidated_codes": [
                    "claim_decomposition",
                    "claim element decomposition",
                    "big_claim"
                ],
                "description": "Large claims not properly broken down into elements; elements analyzed without context of full claim",
                "impact": "HIGH - Incomplete element-by-element analysis",
                "examples": [
                    "Long claim elements are not delimited properly, not analyzed for differences",
                    "Breaking down big claim elements into sentences for better mapping"
                ]
            },
            "Preprocessing & Input Step Recognition": {
                "consolidated_codes": [
                    "pre_processing_mapping",
                    "pre_processing  step",
                    "pre+post_processing_steps"
                ],
                "description": "Failure to correctly classify preprocessing/postprocessing steps; treating them as novel when they are foundational",
                "impact": "HIGH - Incorrect novelty/inventive step assessment",
                "examples": [
                    "Input steps won't have novelty/inventive step - they're just preprocessing",
                    "Postprocessing marked as novel which is obvious"
                ]
            },
            "Patent Document Coverage": {
                "consolidated_codes": [
                    "preamble_missed"
                ],
                "description": "Failure to analyze all sections of patent (preamble, claims, specification)",
                "impact": "MEDIUM - Incomplete analysis",
                "examples": [
                    "Preamble is missed for mapping in all traces"
                ]
            }
        }
    },
    "STAGE_2_CLAIM_ANALYSIS": {
        "name": "Claim Language & Term Analysis",
        "description": "Failures in interpreting claim language, identifying corresponding text between patents",
        "categories": {
            "Term Interpretation & Consistency": {
                "consolidated_codes": [
                    "term_consistent",
                    "verbatim_term"
                ],
                "description": "Treating claim terms verbatim instead of using specification context; inconsistent interpretation",
                "impact": "HIGH - Misinterpretation of claim scope",
                "examples": [
                    "Using specification support to understand claim terms better rather than verbatim",
                    "Element mapping inconsistent across analysis"
                ]
            },
            "Corresponding Text Identification": {
                "consolidated_codes": [
                    "correspond_text"
                ],
                "description": "Verdict summarizes/paraphrases rather than quoting actual corresponding text from patent",
                "impact": "MEDIUM - Traceability and verification issues",
                "examples": [
                    "Verdict summarizes the text rather than quoting the corresponding text"
                ]
            }
        }
    },
    "STAGE_3_NOVELTY_ASSESSMENT": {
        "name": "Novelty Assessment",
        "description": "Failures in understanding novelty concept and assessing against prior art",
        "categories": {
            "Novelty vs. Inventive Step Distinction": {
                "consolidated_codes": [
                    "diff_nov_and_invent",
                    "diff_in_tech"
                ],
                "description": "Confusion between novelty and inventive step; missing explanation of technical differences",
                "impact": "HIGH - Fundamental conceptual error",
                "examples": [
                    "Different between novelty and inventive step - need clear distinction",
                    "Explanation of technical difference not provided"
                ]
            },
            "Person Skilled in Art Consideration": {
                "consolidated_codes": [
                    "person_skilled"
                ],
                "description": "Failure to consider assessment from perspective of person skilled in the art",
                "impact": "HIGH - Fundamental patent law requirement missed",
                "examples": [
                    "Inventive step in light of person skilled in art not properly considered"
                ]
            }
        }
    },
    "STAGE_4_INVENTIVE_STEP_ASSESSMENT": {
        "name": "Inventive Step Assessment",
        "description": "All errors related to assessing whether differences from prior art constitute inventive step",
        "categories": {
            "Inventive Step Reasoning & Judgment": {
                "consolidated_codes": [
                    "hallucination_inventive_step",
                    "narrow_inventive_step",
                    "mark_inventive_step",
                    "inventive_why?"
                ],
                "description": "Incorrect marking of inventive step; hallucinated reasoning; threshold too narrow/broad; missing justification for why difference is inventive",
                "impact": "CRITICAL - Direct impact on final verdict",
                "examples": [
                    "Hallucination: says no inventive step but marked as inventive",
                    "Threshold too narrow - slight changes marked as inventive",
                    "Missing explanation of why the difference constitutes an inventive step",
                    "Prior art difference mentioned but not explained why it's inventive"
                ]
            }
        }
    }
}

print("Failure Taxonomy Structure:")
print("="*70)
for stage_key, stage_data in failure_taxonomy.items():
    print(f"\n{stage_key}")
    print(f"  Name: {stage_data['name']}")
    print(f"  Categories: {len(stage_data['categories'])}")
    for cat_name, cat_data in stage_data['categories'].items():
        print(f"    - {cat_name} ({len(cat_data['consolidated_codes'])} codes)")

## Step 4: Map Open Codes to Consolidated Categories

In [ ]:
# Create reverse mapping: open code -> (stage, category, consolidated)
code_to_category = {}

for stage_key, stage_data in failure_taxonomy.items():
    for cat_name, cat_data in stage_data['categories'].items():
        consolidated_name = cat_name
        for open_code in cat_data['consolidated_codes']:
            code_to_category[open_code.lower().strip()] = {
                'stage': stage_key,
                'stage_name': stage_data['name'],
                'category': cat_name,
                'impact': cat_data['impact']
            }

print("Code to Category Mapping:")
print("="*70)
for code in sorted(code_to_category.keys()):
    mapping = code_to_category[code]
    print(f"\n'{code}'")
    print(f"  → Stage: {mapping['stage_name']}")
    print(f"  → Category: {mapping['category']}")
    print(f"  → Impact: {mapping['impact']}")

## Step 5: Frequency Analysis by Category

In [ ]:
# Count failures by category
category_counts = defaultdict(int)
stage_counts = defaultdict(int)
category_to_modes = defaultdict(list)

for ann in filtered_annotations:
    modes = ann.get('open_coded_failure_modes', [])
    for mode in modes:
        normalized = mode.strip().lower()
        if normalized in code_to_category:
            mapping = code_to_category[normalized]
            stage = mapping['stage_name']
            category = mapping['category']
            
            stage_counts[stage] += 1
            category_counts[f"{stage} > {category}"] += 1
            category_to_modes[f"{stage} > {category}"].append(normalized)

print("\nFailure Counts by Stage:")
print("="*70)
for stage in sorted(stage_counts.keys()):
    print(f"{stage:45} : {stage_counts[stage]:3d} occurrences")

print(f"\nTotal failures (excluding token_limit): {sum(stage_counts.values())}")

print("\n\nFailure Counts by Category:")
print("="*70)
for category in sorted(category_counts.keys(), key=lambda x: category_counts[x], reverse=True):
    count = category_counts[category]
    print(f"{count:3d} | {category}")

## Step 6: Visualization

In [ ]:
# Plot failures by stage
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Stage distribution
stages = list(stage_counts.keys())
counts = list(stage_counts.values())
stage_labels = [s.replace('STAGE_', '').replace('_', ' ') for s in stages]

axes[0].barh(stage_labels, counts, color='steelblue')
axes[0].set_xlabel('Number of Failures')
axes[0].set_title('Failure Count by Workflow Stage')
axes[0].grid(axis='x', alpha=0.3)

# Verdict distribution
verdicts = [pass_count, fail_count]
verdict_labels = ['PASS', 'FAIL']
axes[1].pie(verdicts, labels=verdict_labels, autopct='%1.1f%%', colors=['green', 'red'])
axes[1].set_title(f'Overall Verdict Distribution (n={len(filtered_annotations)})')

plt.tight_layout()
plt.savefig('phase2_failure_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("Chart saved: phase2_failure_analysis.png")

## Step 7: Generate Failure Taxonomy JSON

In [ ]:
# Create comprehensive taxonomy JSON for Phase 3 re-annotation
taxonomy_output = {
    "phase": 2,
    "date_generated": "2026-04-25",
    "methodology": "Axial Coding - Workflow Stage Based",
    "statistics": {
        "total_traces_analyzed": len(filtered_annotations),
        "pass_traces": pass_count,
        "fail_traces": fail_count,
        "total_failures_identified": sum(stage_counts.values()),
        "unique_consolidated_categories": len(category_counts)
    },
    "exclusions": {
        "technical_errors": ["token_limit"],
        "reason": "Treated as system/environmental constraint, not methodology error"
    },
    "failure_taxonomy": failure_taxonomy,
    "stage_summary": {
        stage: {
            "count": stage_counts[stage],
            "percentage": f"{100*stage_counts[stage]/sum(stage_counts.values()):.1f}%"
        }
        for stage in sorted(stage_counts.keys())
    }
}

# Save taxonomy
with open('failure_taxonomy.json', 'w') as f:
    json.dump(taxonomy_output, f, indent=2)

print("Taxonomy saved to: failure_taxonomy.json")
print(f"\nTaxonomy Summary:")
print(json.dumps(taxonomy_output['statistics'], indent=2))

## Step 8: Ready for Phase 3 - Re-annotation with Standardized Categories

In [ ]:
print("\n" + "="*70)
print("PHASE 2 COMPLETE: FAILURE TAXONOMY GENERATED")
print("="*70)

print("\nNext Steps (Phase 3 - Re-annotation):")
print("""
1. Load failure_taxonomy.json in the annotation tool (Phase 3 mode)
2. For each trace, re-annotate using consolidated categories instead of open codes
3. Output: traces_annotations.jsonl with 'failure_modes' field (standardized)
4. Maintain 'open_coded_failure_modes' field for traceability

Workflow Stages for Re-annotation:
  1. Input Understanding & Decomposition (4 failure types)
  2. Claim Language & Term Analysis (2 failure types)
  3. Novelty Assessment (2 failure types)
  4. Inventive Step Assessment (1 consolidated failure type)

Total Consolidated Categories: 9
Total Failures Analyzed: {}
Ready for Phase 3 refinement and evaluation.
""".format(sum(stage_counts.values())))

## Appendix: Detailed Taxonomy Summary

In [ ]:
# Print detailed taxonomy for reference
print("\nDETAILED FAILURE TAXONOMY")
print("="*80)

for stage_key in sorted(failure_taxonomy.keys()):
    stage_data = failure_taxonomy[stage_key]
    stage_count = stage_counts.get(stage_data['name'], 0)
    
    print(f"\n{'='*80}")
    print(f"STAGE: {stage_data['name']} ({stage_count} failures)")
    print(f"{'='*80}")
    print(f"Description: {stage_data['description']}\n")
    
    for cat_name, cat_data in stage_data['categories'].items():
        print(f"\n  CATEGORY: {cat_name}")
        print(f"  Impact: {cat_data['impact']}")
        print(f"  Description: {cat_data['description']}")
        print(f"  Open Codes: {', '.join(cat_data['consolidated_codes'])}")
        print(f"  Examples:")
        for example in cat_data['examples']:
            print(f"    - {example}")